##### HIGH LEVEL IMPLEMENTATION OF THE PAPER 'Sequence to Sequence Learning with Neural Networks (Sutskever et al., 2014)' USING PYTORCH

In [35]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import re
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import random
import time
import math


#### if you are using google colab , uncomment the next section to download the data

In [ ]:
# if you are using google colab , uncomment this section to download the data
# import urllib.request
# import zipfile
# import os

# # Download a small, clean dataset of English-German pairs
# url = "https://www.manythings.org/anki/deu-eng.zip"
# zip_path = "deu-eng.zip"

# # Create a request with a User-Agent header to avoid HTTP Error 406
# req = urllib.request.Request(
#     url,
#     headers={
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
#     },
# )

# # Open the URL and retrieve the data
# with urllib.request.urlopen(req) as response:
#     with open(zip_path, 'wb') as out_file:
#         out_file.write(response.read())

# # Extract it
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(".")

In [36]:
with open("deu.txt", "r", encoding="utf-8") as f:
    lines = f.read().strip().split("\n")
# Keep a subset of 10,000 short sentences to save memory and train fast
pairs = []
for line in lines[0:3]:
    # The file is tab-separated: English \t German \t Attribution
    parts = line.split("\n")[0].split("\t")
    pairs.append((parts[0], parts[1]))

print(f"Sample pair: {pairs[:3]}")

Sample pair: [('Go.', 'Geh.'), ('Hi.', 'Hallo!'), ('Hi.', 'Grüß Gott!')]


In [ ]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.num_words = 4

    def add_sentence(self, sentence):
        sentence = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", sentence.lower())
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.num_words
                self.idx2word[self.num_words] = word
                self.num_words += 1

    def numericalize(self, sentence):
        sentence = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", sentence.lower())
        return [
            self.word2idx.get(word, self.word2idx["<UNK>"]) for word in sentence.split()
        ]


# Initialize separate vocabularies for Source and Target languages
eng_vocab = Vocabulary()
deu_vocab = Vocabulary()

for eng, deu in pairs:
    eng_vocab.add_sentence(eng)
    deu_vocab.add_sentence(deu)

In [38]:
type(pairs[0])

tuple

In [39]:
print(pairs)

[('Go.', 'Geh.'), ('Hi.', 'Hallo!'), ('Hi.', 'Grüß Gott!')]


In [40]:
print(eng_vocab)

In [41]:
print(eng_vocab.word2idx)
print(eng_vocab.idx2word)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'go': 4, 'hi': 5}
{0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>', 4: 'go', 5: 'hi'}


In [42]:
print(deu_vocab.word2idx)
print(deu_vocab.idx2word)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'geh': 4, 'hallo': 5, 'grüß': 6, 'gott': 7}
{0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>', 4: 'geh', 5: 'hallo', 6: 'grüß', 7: 'gott'}


In [43]:
print(eng_vocab.numericalize("hi, go out "))

[5, 4, 3]


In [44]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        print(">>>>>>>>>>>initializing Tranlation dataset <<<<<<<<<<<<<<<<<<")
        print("length of pairs")
        print(len(self.pairs))
        print("pairs:")
        print(self.pairs)
        print("--------------")
        print("src vocab")
        print(self.src_vocab.word2idx)
        print("--------------")
        print("trg vocab")
        print(self.trg_vocab.word2idx)
        print(
            "---------------------- end of Translation dataset initialization -----------------------------"
        )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        print(">>>>>>>>>>>starting: getting items in dataset <<<<<<<<<<<<<<<<<<")
        print("index:", idx)
        src_sent, trg_sent = self.pairs[idx]
        # Numericalize text strings
        src_tokens = self.src_vocab.numericalize(src_sent)
        trg_tokens = self.trg_vocab.numericalize(trg_sent)
        print("src sentence: ", src_sent)
        print("src token: ", src_tokens)
        print("----------")
        print("trg sentence: ", trg_sent)
        print("trg tokens", trg_tokens)
        print("-----------")
        # paper recommendation
        src_tokens = src_tokens[::-1]
        print("source reversed:", src_tokens)
        src_tensor = torch.tensor(
            src_tokens + [self.src_vocab.word2idx["<EOS>"]], dtype=torch.long
        )
        print("----------")
        print("src shape: ", src_tensor.shape)
        print("src tensor: ", src_tensor)
        trg_tensor = torch.tensor(
            [self.trg_vocab.word2idx["<SOS>"]]
            + trg_tokens
            + [self.trg_vocab.word2idx["<EOS>"]],
            dtype=torch.long,
        )
        print("----------")
        print("trg shape: ", trg_tensor.shape)
        print("trg tensor:")
        print(trg_tensor)
        print(
            "----------------------------------end of get item----------------------------"
        )
        return src_tensor, trg_tensor


In [45]:
# Collate function to dynamically pad batches to equivalent lengths
def collate_fn(batch):
    print(">>>>>>>>>>> collate_fn printing batch <<<<<<<<<<<<")
    print("current batch: ")
    print(batch)
    print("----------")
    src_batch, trg_batch = zip(*batch)
    print("collate_fn printing src and tg before padding......")
    print(src_batch)
    print(trg_batch)
    print("----------")
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)
    print("collate_fn printing src batch after padding ...")
    print(src_batch)
    print(trg_batch)
    print(
        "----------------------------end of collate_fin---------------------------------------"
    )
    return src_batch, trg_batch


In [46]:
dataset = TranslationDataset(pairs, eng_vocab, deu_vocab)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

>>>>>>>>>>>initializing Tranlation dataset <<<<<<<<<<<<<<<<<<
length of pairs
3
pairs:
[('Go.', 'Geh.'), ('Hi.', 'Hallo!'), ('Hi.', 'Grüß Gott!')]
--------------
src vocab
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'go': 4, 'hi': 5}
--------------
trg vocab
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'geh': 4, 'hallo': 5, 'grüß': 6, 'gott': 7}
---------------------- end of Translation dataset initialization -----------------------------


In [47]:
ss = torch.tensor([19, 99, 483, 132, 1275, 3], dtype=torch.long)
print(ss, ss.shape)

tensor([  19,   99,  483,  132, 1275,    3]) torch.Size([6])


In [48]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers):
        print(">>>>>>>>>>>initializing starting encoder")
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        print("inputs:")
        print("input dim: ", input_dim)
        print("embedding dim: ", emb_dim)
        print("hidden dim: ", hidden_dim)
        print("layers: ", n_layers)
        print("embedding")
        print(self.embedding)
        # Using PyTorch's native deep multi-layer LSTM cell wrapper
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, batch_first=True)
        print("LSTM")
        print(self.rnn)

    def forward(self, src):
        print(">>>>> encoder forward starting ------")
        print("source shape :", src.shape)
        print("source :", src)
        embedded = self.embedding(src)
        print("embedding shape: ", embedded.shape)
        print("embedding ")

        print(embedded)
        # We ignore intermediate sequence outputs; we only extract final context hidden states
        all_outputs, (hidden, cell) = self.rnn(embedded)
        print("outputs:")
        print("all_output shape: ", all_outputs.shape)
        print("all_output: ", all_outputs)

        print("hidden shape", hidden.shape)
        print("hidden: ", hidden)

        print("cell shape ", cell.shape)
        print("cell: ", cell)
        print(
            "-------------------------------- encoder foward ends -----------------------"
        )
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers):
        print(">>>>>> starting decoder intitalization")
        super().__init__()
        print("parameters ")
        print("output dim: ", output_dim)
        print("embedding dim:", emb_dim)
        print("hidden dim: ", hidden_dim)
        print("layers :", n_layers)
        self.embedding = nn.Embedding(output_dim, emb_dim)
        print("embedding :")
        print(self.embedding.embedding_dim)
        print(self.embedding)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, batch_first=True)
        print("LSTM")
        print(self.rnn)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        print("Linear output")
        print(self.fc_out)

    def forward(self, input_token, hidden, cell):
        print(">>>>> decoder forward starting")
        print("parameters: ")
        print("input token shape: ", input_token.shape)
        print("input token: ", input_token)
        print("cell shape ", cell.shape)
        print("cell: ", cell)
        print("hidden shape", hidden.shape)
        print("hidden ", hidden)
        # Input token shape: (batch_size) -> unsqueeze to track sequence steps (batch_size, 1)
        input_token = input_token.unsqueeze(1)
        print("unsqueezed input token shape ", input_token.shape)
        print("unsqueezed input token: ", input_token)
        embedded = self.embedding(input_token)
        print("embedding shape: ", embedded.shape)
        print("embedding: ", embedded)

        # Process the single sequence step using preceding context records
        all_output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        print("outputs: ")
        print("all_output shape ", all_output.shape)
        print("all_output: ", all_output)

        print("cell shape ", cell.shape)
        print("cell: ", cell)
        print("hidden shape", hidden.shape)
        print("hidden ", hidden)
        prediction = self.fc_out(all_output.squeeze(1))
        print("prediction shape", prediction.shape)
        print("prediction: ", prediction)
        print("------------------------------- decoder ends -------------------------")
        return prediction, hidden, cell

In [49]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        print(">>>>>> starting Seq2Seq module")
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        print("params:")
        print("encoder: ", encoder)
        print("decoder: ", decoder)
        print("device: ", device)

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        print(">>>>>> seq2seq forward starting")
        print("parameters")
        print("src shape: ", src.shape)
        print("src: ", src)
        print("trg shape: ", trg.shape)
        print("trg: ", trg)
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features
        # print("decoder shape: ", self.decoder.fc_out.shape)
        print("vocab size", trg_vocab_size)
        # Tensor vector space shell to accumulate individual sequence element predictions
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        print("output shape: ", outputs.shape)
        # Capture fixed vector representations from the Encoder segment
        hidden, cell = self.encoder(src)
        print("cell shape ", cell.shape)
        print("cell: ", cell)
        print("hidden shape", hidden.shape)
        print("hidden ", hidden)
        # Initialize output loop step with the sequence starting boundary signal
        input_token = trg[:, 0]
        print("input token shape: ", input_token.shape)
        print("input shape: ", input_token)
        for t in range(1, trg_len):
            print("index :", t)
            prediction, hidden, cell = self.decoder(input_token, hidden, cell)
            print("prediction shape:", prediction.shape)
            print("prediction: ", prediction)
            print("hidden shape", hidden.shape)
            print("cell shape", cell.shape)
            outputs[:, t, :] = prediction

            # Determine teacher forcing strategy for the next input step
            teacher_force = random.random() < teacher_forcing_ratio
            print("teacher force:", teacher_force)

            top1 = prediction.argmax(1)
            print("top1: ", top1)
            input_token = trg[:, t] if teacher_force else top1
            print("input token shape: ", input_token.shape)
            print("input token: ", input_token)
        return outputs

In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scale dimensions way down relative to the 1000-cell, 4-layer paper parameters to fit your Colab notebook
INPUT_DIM = eng_vocab.num_words
OUTPUT_DIM = deu_vocab.num_words
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HIDDEN_DIM = 256
N_LAYERS = 2

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM, N_LAYERS)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM, N_LAYERS)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)  # Ignore <PAD> tokens in loss calculation

print(f"Model initialized successfully. Device: {device}")

>>>>>>>>>>>initializing starting encoder
inputs:
input dim:  6
embedding dim:  128
hidden dim:  256
layers:  2
embedding
Embedding(6, 128)
LSTM
LSTM(128, 256, num_layers=2, batch_first=True)
>>>>>> starting decoder intitalization
parameters 
output dim:  8
embedding dim: 128
hidden dim:  256
layers : 2
embedding :
128
Embedding(8, 128)
LSTM
LSTM(128, 256, num_layers=2, batch_first=True)
Linear output
Linear(in_features=256, out_features=8, bias=True)
>>>>>> starting Seq2Seq module
params:
encoder:  Encoder(
  (embedding): Embedding(6, 128)
  (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
)
decoder:  Decoder(
  (embedding): Embedding(8, 128)
  (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc_out): Linear(in_features=256, out_features=8, bias=True)
)
device:  cpu
Model initialized successfully. Device: cpu


In [51]:
def train(model, dataloader, optimizer, criterion, clip):
    print(">>>>>>>>>>> starting training looopp. ")
    print("parameters")
    print("model: ", model)
    print("dataloader: ", dataloader)
    print("optimizer: ", optimizer)
    print("criterion: ", criterion)
    print("clip: ", clip)
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(dataloader):
        print("training loop")
        print("index: ", i)
        print("src: ", src)
        print("trg: ", trg)
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()
        print("optimizer: ", optimizer)
        # Forward pass: pass data through the model layout
        output = model(src, trg)
        print("output: ", output.shape)
        print("output: ", output)
        # trg = [batch size, trg len]
        # output = [batch size, trg len, output dim]

        output_dim = output.shape[-1]
        print("output_dim", output_dim)
        # Reshape output and target tensors to compute cross-entropy loss
        output = output[:, 1:].reshape(-1, output_dim)
        print("output shape: ", output.shape)
        print("output: ", output)
        trg = trg[:, 1:].reshape(-1)
        print("trg shape: ", trg.shape)
        print("trg: ", trg)

        loss = criterion(output, trg)
        print("loss : ", loss)
        loss.backward()

        # Gradient clipping - a standard trick for training stable LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


# Run a quick training cycle test
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    start_time = time.time()
    train_loss = train(model, dataloader, optimizer, criterion, CLIP)
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )

>>>>>>>>>>> starting training looopp. 
parameters
model:  Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(6, 128)
    (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(8, 128)
    (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
    (fc_out): Linear(in_features=256, out_features=8, bias=True)
  )
)
dataloader:  <torch.utils.data.dataloader.DataLoader object at 0x128f66330>
optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
criterion:  CrossEntropyLoss()
clip:  1
>>>>>>>>>>>starting: getting items in dataset <<<<<<<<<<<<<<<<<<
index: 0
src sentence:  Go.
src token:  [4]
----------
trg sentence:  Geh.
trg tokens [4]
-----------
source reversed: [4]
----------
src shape:  torch.Size([2])
src tensor:  tensor([4, 2])
----------
trg shape:  t